In [ ]:
import json
from datetime import datetime, date, time, timedelta

# ---------------------------------------------------------------------------
#This script doesn't use functions -- each transformation that would've been a function call is just hardcoded in.

breaks = [
    (time(9, 0), time(9, 15)),
    (time(13, 0), time(13, 30)),
    (time(14, 0), time(14, 15)),
    (time(17, 0), time(17, 15)),
    (time(19, 0), time(19, 30)),
    (time(23, 0), time(23, 15)),
]
holidays = {date(2026, 7, 15)}

start = datetime(2026, 7, 1, 0, 0)
end = datetime(2026, 7, 22, 0, 0)
step = timedelta(hours=0.01)

ticks = []
anchor = start
while anchor < end:
    tod = anchor.time()

    open_tick = True
    if time(1, 0) <= tod < time(7, 0):
        open_tick = False
    else:
        shift_day = anchor.date() - timedelta(days=1) if tod < time(1, 0) else anchor.date()
        if shift_day.weekday() >= 5 or shift_day in holidays:
            open_tick = False
        else:
            for b_start, b_end in breaks:
                if b_start <= tod < b_end:
                    open_tick = False
                    break

    if open_tick:
        ticks.append(anchor)

    anchor += step

# --- load data ---

with open("cnc_and_weld_jobs.json") as f:
    jobs = json.load(f)

with open("shop_config.json") as f:
    work_centers = json.load(f)

DEPT_MAP = {
    1: work_centers["dept_1_work_centers"],
    2: work_centers["dept_2_work_centers"],
}

# --- schedule_jobs() + qualifies(), inlined ---
jobs_sorted = sorted(jobs, key=lambda j: j["due_date"])

wc_cursor = {}
for dept in DEPT_MAP.values():
    for wc in dept:
        wc_cursor[wc["Number"]] = 0

results = []

for job in jobs_sorted:
    op_ready_index = 0
    op_results = []
    unscheduled = False

    for op in sorted(job["operations"], key=lambda o: o["op_seq"]):
        department = op["department"]
        material = job["stock_material"]

        # qualifies(), inlined as an explicit loop instead of a comprehension
        candidates = []
        for wc in DEPT_MAP[department]:
            if department == 1:
                key = "Steel" if material == "steel" else "Aluminum"
            elif department == 2:
                key = "Weld_Steel" if material == "steel" else "Weld_Aluminum"
            else:
                key = None
            if key is not None and wc.get(key, False):
                candidates.append(wc)

        if not candidates:
            op_results.append({
                "op_seq": op["op_seq"],
                "status": f"no work center qualifies (dept {department}, {material})",
            })
            unscheduled = True
            break

        duration_ticks = round((op["setup_time"] + op["run_time"]) * 100)

        best_wc, best_start = None, None
        for wc in candidates:
            wc_num = wc["Number"]
            start_index = max(op_ready_index, wc_cursor[wc_num])
            if best_start is None or start_index < best_start:
                best_start, best_wc = start_index, wc_num

        end_index = best_start + duration_ticks

        if end_index > len(ticks):
            op_results.append({
                "op_seq": op["op_seq"],
                "status": "unscheduled - runs past end of calendar window",
            })
            unscheduled = True
            break

        start_time = ticks[best_start]
        finish_time = ticks[end_index - 1]

        wc_cursor[best_wc] = end_index
        op_ready_index = end_index

        op_results.append({
            "op_seq": op["op_seq"],
            "work_center": best_wc,
            "start": start_time,
            "finish": finish_time,
            "duration_ticks": duration_ticks,
        })

    due = date.fromisoformat(job["due_date"])
    first_start = op_results[0].get("start") if op_results else None
    last_finish = op_results[-1].get("finish") if op_results else None
    late = (last_finish.date() > due) if last_finish else None

    results.append({
        "job_number": job["job_number"],
        "due_date": due,
        "start": first_start,
        "finish": last_finish,
        "late": late,
        "unscheduled": unscheduled,
        "operations": op_results,
    })

# --- write_outputs(), inlined ---
import pandas as pd

summary_path = "schedule_summary_flat.csv"
detail_path = "schedule_detail_flat.csv"

summary_rows = []
detail_rows = []

for r in results:
    summary_rows.append({
        "job_number": r["job_number"],
        "start": r["start"].strftime("%Y-%m-%d %H:%M") if r["start"] else None,
        "due_date": r["due_date"],
        "expected_finish": r["finish"].strftime("%Y-%m-%d %H:%M") if r["finish"] else None,
        "status": "LATE" if r["late"] else ("ON TIME" if r["late"] is False else "UNSCHEDULED"),
    })

    for op in r["operations"]:
        row = {"job_number": r["job_number"], "op_seq": op["op_seq"]}
        if "status" in op:
            row["status"] = op["status"]
        else:
            row.update({
                "work_center": op["work_center"],
                "start": op["start"].strftime("%Y-%m-%d %H:%M"),
                "finish": op["finish"].strftime("%Y-%m-%d %H:%M"),
            })
        detail_rows.append(row)

pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
pd.DataFrame(detail_rows).to_csv(detail_path, index=False)
print(f"Wrote {summary_path} ({len(summary_rows)} jobs)")
print(f"Wrote {detail_path} ({len(detail_rows)} operations)")